# Manufacturing Defect Classification

**Tabular Classification Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `defective`

## 2 · Project Overview

We predict whether a manufactured item is **defective** based on process parameters: temperature, pressure, humidity, vibration, speed, material grade, and production shift.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular classification dataset.
2. Encode categorical features for tree-based models.
3. Build a baseline Logistic Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given manufacturing process parameters (temperature, pressure, vibration, speed, material grade, shift), predict whether the produced item will be defective.

## 5 · Why This Project Matters

- **Quality control** is critical in manufacturing — defects cost money and reputation.
- Predictive quality enables proactive process adjustment.
- Understanding defect drivers improves process design.
- Directly reduces scrap, rework, and warranty costs.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 8,000 |
| **Features** | temperature, pressure, humidity, vibration, speed_rpm, material_grade, shift |
| **Target** | defective (0 = good, 1 = defective) |
| **Class balance** | ~85/15 |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "defective"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

Target: defective
Save dir: E:\Github\Machine-Learning-Projects\Classification\Manufacturing Defect Classification


## 11 · Dataset Download or Loading

Synthetic dataset: 8,000 manufactured items with process parameters and defect outcome.

In [4]:
np.random.seed(SEED)
n = 8000
temperature = np.round(np.random.normal(200, 15, n), 1)
pressure = np.round(np.random.normal(50, 8, n), 1)
humidity = np.round(np.random.uniform(30, 80, n), 1)
vibration = np.round(np.random.exponential(2, n).clip(0, 15), 2)
speed_rpm = np.round(np.random.normal(3000, 300, n), 0)
material_grade = np.random.choice(["A", "B", "C"], n, p=[0.4, 0.35, 0.25])
grade_num = np.where(material_grade == "A", 2, np.where(material_grade == "B", 1, 0))
shift = np.random.choice(["Day", "Evening", "Night"], n, p=[0.4, 0.35, 0.25])
shift_num = np.where(shift == "Day", 0, np.where(shift == "Evening", 1, 2))

# Defects more likely at extreme temperatures, high vibration, night shift, low grade
score = (0.02 * np.abs(temperature - 200) + 0.15 * vibration - 0.01 * pressure
         - 0.2 * grade_num + 0.15 * shift_num + 0.005 * np.abs(speed_rpm - 3000)
         + np.random.normal(0, 0.7, n))
defective = (score > np.percentile(score, 85)).astype(int)

df = pd.DataFrame({"temperature": temperature, "pressure": pressure,
                    "humidity": humidity, "vibration": vibration,
                    "speed_rpm": speed_rpm, "material_grade": material_grade,
                    "shift": shift, "defective": defective})
print(f"Dataset shape: {df.shape}")
print(f"Class balance:\n{df['defective'].value_counts(normalize=True).round(3)}")
df.head()

Dataset shape: (8000, 8)
Class balance:
defective
0    0.85
1    0.15
Name: proportion, dtype: float64


,temperature,pressure,humidity,vibration,speed_rpm,material_grade,shift,defective
0,207.5,49.7,55.9,2.64,3203.0,A,Night,0
1,197.9,46.0,75.6,2.57,2990.0,A,Night,0
2,209.7,48.6,46.3,1.61,3357.0,A,Day,0
3,222.8,55.7,37.8,0.55,2615.0,A,Night,0
4,196.5,60.2,65.4,0.59,2669.0,C,Evening,1


## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed. Value counts:")
print(df[TARGET].value_counts())

DATA VALIDATION

Shape: (8000, 8)

Missing values:
temperature       0
pressure          0
humidity          0
vibration         0
speed_rpm         0
material_grade    0
shift             0
defective         0
dtype: int64

Duplicate rows: 0

Dtypes:
temperature       float64
pressure          float64
humidity          float64
vibration         float64
speed_rpm         float64
material_grade     object
shift              object
defective           int64
dtype: object

Target 'defective' confirmed. Value counts:
defective
0    6800
1    1200
Name: count, dtype: int64


## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [6]:
num_cols = ["temperature", "pressure", "humidity", "vibration", "speed_rpm"]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, col in enumerate(num_cols):
    ax = axes[i // 3][i % 3]
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col)
axes[1][2].axis("off")
plt.suptitle("Feature Distributions by Defect Status", fontsize=14)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df.groupby("material_grade")[TARGET].mean().plot(kind="bar", ax=axes[0], color="steelblue", edgecolor="black")
axes[0].set_title("Defect Rate by Material Grade")
df.groupby("shift")[TARGET].mean().plot(kind="bar", ax=axes[1], color="steelblue", edgecolor="black")
axes[1].set_title("Defect Rate by Shift")
plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `defective`.

In [7]:
fig, ax = plt.subplots(figsize=(6, 4))
df[TARGET].value_counts().plot(kind="bar", ax=ax, color=["steelblue", "salmon"], edgecolor="black")
ax.set_title("Defect Distribution")
ax.set_xticklabels(["Good (0)", "Defective (1)"], rotation=0)
plt.tight_layout()
plt.show()
print(f"Defect rate: {df[TARGET].mean():.1%}")

Defect rate: 15.0%


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 train/test split preserving class balance.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

# Encode target if needed
if y.dtype == object:
    le = LabelEncoder()
    y = pd.Series(le.fit_transform(y), name=TARGET)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train class distribution:\n{y_train.value_counts(normalize=True).round(3)}")

Train: (6400, 7), Test: (1600, 7)
Train class distribution:
defective
0    0.85
1    0.15
Name: proportion, dtype: float64


## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `material_grade`, `shift` encoded with OrdinalEncoder.
- **Scaling**: Not needed for tree models.
- **Class imbalance**: ~15% defective — moderate imbalance, stratified split used.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [9]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["temp_deviation"] = np.abs(X_train["temperature"] - 200)
X_test["temp_deviation"] = np.abs(X_test["temperature"] - 200)

X_train["speed_deviation"] = np.abs(X_train["speed_rpm"] - 3000)
X_test["speed_deviation"] = np.abs(X_test["speed_rpm"] - 3000)

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (9): ['temperature', 'pressure', 'humidity', 'vibration', 'speed_rpm', 'material_grade', 'shift', 'temp_deviation', 'speed_deviation']


## 18 · Baseline Model

Logistic Regression as our baseline classifier.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

acc_bl = accuracy_score(y_test, y_pred_bl)
f1_bl = f1_score(y_test, y_pred_bl, average="binary")

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {acc_bl:.4f}")
print(f"F1       : {f1_bl:.4f}")
print("\n" + classification_report(y_test, y_pred_bl))

BASELINE: Logistic Regression
Accuracy : 0.9181
F1       : 0.6888

              precision    recall  f1-score   support

           0       0.93      0.97      0.95      1360
           1       0.80      0.60      0.69       240

    accuracy                           0.92      1600
   macro avg       0.87      0.79      0.82      1600
weighted avg       0.91      0.92      0.91      1600



## 19 · LazyPredict Benchmark

Fit dozens of classifiers in one call for a quick ranking.

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                               Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                          
NearestCentroid                0.868750           0.867892  0.942218  0.880333   0.908424  0.868750    0.016554
QuadraticDiscriminantAnalysis  0.912500           0.826716  0.946645  0.912349   0.912203  0.912500    0.016682
GaussianNB                     0.908750           0.815931  0.944078  0.908273   0.907837  0.908750    0.019109
LinearDiscriminantAnalysis     0.915000           0.797304  0.948915  0.911571   0.910435  0.915000    0.022220
LGBMClassifier                 0.916250           0.794608  0.938009  0.912323   0.911470  0.916250    3.237704
LogisticRegression             0.919375           0.793015  0.950132  0.914770   0.914705  0.919375    0.019746
CalibratedClassifierCV         0.918125           0.788848  0.950070  

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="classification", time_budget=60,
                 metric="f1",
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1: {f1_score(y_test, y_pred_flaml, average='binary'):.4f}")

FLAML best model: catboost
Accuracy: 0.9169
F1: 0.6826


## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [13]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostClassifier
    t0 = time.perf_counter()
    try:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test).ravel()
    print(f"CatBoost F1: {f1_score(y_test, results['CatBoost'], average='binary'):.4f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM F1: {f1_score(y_test, results['LightGBM'], average='binary'):.4f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBClassifier
    t0 = time.perf_counter()
    try:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost F1: {f1_score(y_test, results['XGBoost'], average='binary'):.4f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost F1: 0.6825  (5.3s)


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[111]	valid_0's binary_logloss: 0.204454
LightGBM F1: 0.6759  (6.8s)


XGBoost failed: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'


## 22 · Top 3 Model Selection

Rank all models by F1 and select the top 3.

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average="binary"), 4),
        "Precision": round(precision_score(y_test, yp, average="binary", zero_division=0), 4),
        "Recall": round(recall_score(y_test, yp, average="binary", zero_division=0), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
Logistic Regression    0.9181  0.6888     0.8011  0.6042
FLAML                  0.9169  0.6826     0.7989  0.5958
CatBoost               0.9163  0.6825     0.7912  0.6000
LightGBM               0.9125  0.6759     0.7604  0.6083

Top 3 models: ['Logistic Regression', 'FLAML', 'CatBoost']


## 23 · Final Training and Evaluation of Top 3

Detailed metrics and confusion matrices.

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap="Blues")
    axes[i].set_title(f"{name}\nF1={f1_score(y_test, yp, average='binary'):.4f}")

plt.suptitle("Top 3 Models — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_confusion_matrices.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    Accuracy : {accuracy_score(y_test, yp):.4f}")
    print(f"    F1       : {f1_score(y_test, yp, average='binary'):.4f}")
    print(f"    Precision: {precision_score(y_test, yp, average='binary', zero_division=0):.4f}")
    print(f"    Recall   : {recall_score(y_test, yp, average='binary', zero_division=0):.4f}")


Detailed Metrics — Top 3:

  Logistic Regression:
    Accuracy : 0.9181
    F1       : 0.6888
    Precision: 0.8011
    Recall   : 0.6042

  FLAML:
    Accuracy : 0.9169
    F1       : 0.6826
    Precision: 0.7989
    Recall   : 0.5958

  CatBoost:
    Accuracy : 0.9163
    F1       : 0.6825
    Precision: 0.7912
    Recall   : 0.6000


## 24 · Error Analysis

Examine misclassifications from the best model.

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]

print(f"Best model: {best_name}")
print(f"\nClassification Report:")
print(classification_report(y_test, best_pred))

errors = X_test.copy()
errors["actual"] = y_test.values
errors["predicted"] = best_pred
errors["correct"] = errors["actual"] == errors["predicted"]

n_errors = (~errors["correct"]).sum()
print(f"\nTotal errors: {n_errors} / {len(errors)} ({n_errors/len(errors)*100:.1f}%)")

if n_errors > 0:
    print("\nSample misclassifications:")
    print(errors[~errors["correct"]].head(10).to_string())

Best model: Logistic Regression

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.97      0.95      1360
           1       0.80      0.60      0.69       240

    accuracy                           0.92      1600
   macro avg       0.87      0.79      0.82      1600
weighted avg       0.91      0.92      0.91      1600


Total errors: 131 / 1600 (8.2%)

Sample misclassifications:
      temperature  pressure  humidity  vibration  speed_rpm  material_grade  shift  temp_deviation  speed_deviation  actual  predicted  correct
6724        192.5      47.3      65.2       0.49     3441.0             2.0    2.0             7.5            441.0       1          0    False
5294        219.8      61.5      35.7       0.70     3301.0             0.0    0.0            19.8            301.0       1          0    False
5961        231.6      52.9      62.8       0.30     3389.0             2.0    1.0            31.6            389.0       1 

## 25 · Interpretation and Business Insight

**Key findings:**
- **Vibration** is the strongest defect predictor.
- **Temperature deviation** from optimal (200°) increases defect risk.
- **Night shift** has higher defect rates.
- **Material grade A** has the lowest defect rate.

**Business takeaway:** Monitor vibration levels, control temperature tightly, and investigate night shift quality issues.

## 26 · Limitations

1. Synthetic data with simplified process physics.
2. No sensor time-series data.
3. Missing tool wear and machine maintenance history.
4. No defect type classification (scratch, crack, dimension).
5. Single-point measurements vs. continuous monitoring.

## 27 · How to Improve This Project

1. Use real sensor data from manufacturing lines.
2. Add tool wear and maintenance features.
3. Include time-series from process sensors.
4. Classify defect types, not just binary.
5. Implement real-time monitoring with streaming models.

## 28 · Production Considerations

- Deploy as real-time quality prediction on the production line.
- Trigger alerts when defect probability exceeds threshold.
- Integrate with SCADA/MES systems.
- Retrain with fresh production data monthly.
- Monitor for process drift (new materials, equipment changes).

## 29 · Common Mistakes

1. Not considering process physics in feature engineering.
2. Using accuracy on imbalanced defect data.
3. Ignoring the cost of false negatives (missed defects reaching customers).
4. Not investigating root causes of defects.
5. Overfitting to specific machine or material batch.

## 30 · Mini Challenge / Exercises

1. Remove `vibration` — how much does recall drop?
2. Create interaction: `temperature * vibration` and test.
3. Try cost-sensitive learning with 10:1 false negative cost.
4. Analyze defects by shift and material grade combination.
5. Plot ROC and PR curves — which is more informative?

## 31 · Final Summary / Key Takeaways

1. **Vibration and temperature deviation** are the strongest defect predictors.
2. **Night shift** shows higher defect rates.
3. **Material grade** significantly affects quality.
4. **Class imbalance** requires precision-recall focus.
5. **Process deviation features** (distance from optimal) are powerful.
6. **Real-time prediction** enables proactive quality control.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average="binary")), 4),
        "precision": round(float(precision_score(y_test, yp, average="binary", zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, yp, average="binary", zero_division=0)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Classification\Manufacturing Defect Classification\metrics.json
{
  "CatBoost": {
    "accuracy": 0.9163,
    "f1": 0.6825,
    "precision": 0.7912,
    "recall": 0.6
  },
  "LightGBM": {
    "accuracy": 0.9125,
    "f1": 0.6759,
    "precision": 0.7604,
    "recall": 0.6083
  },
  "Logistic Regression": {
    "accuracy": 0.9181,
    "f1": 0.6888,
    "precision": 0.8011,
    "recall": 0.6042
  },
  "FLAML": {
    "accuracy": 0.9169,
    "f1": 0.6826,
    "precision": 0.7989,
    "recall": 0.5958
  }
}
